In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:10]:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image

root = '.'
image_dir = '/kaggle/input/images/leftImg8bit_trainvaltest/leftImg8bit/train' 
mask_dir = '/kaggle/input/image-segmentation/gtFine_trainvaltest/gtFine/train'

transform = transforms.Compose([
    transforms.Resize((256,256)),
    #transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
])

class CustomDataset(datasets.ImageFolder):
    def __init__(self, root,img_dir, mask_dir, transform=None):
        self.root = root
        self.img_dir = os.path.join(root,img_dir)
        self.mask_dir = os.path.join(root,mask_dir)
        self.city_dir = os.listdir(self.img_dir)
        
        self.transform = transform
        self.images = []
        self.masks = []
        for city in self.city_dir:
            city_img = os.path.join(self.img_dir,city)
            city_mask = os.path.join(self.mask_dir,city)
            
            for file in os.listdir(city_img):
                img_path = os.path.join(city_img,file)
                mask_path = os.path.join(city_mask,file)
                self.images.append(img_path)
                self.masks.append(mask_path)

    def __getitem__(self, index):
        img_path = self.images[index]
        mask_path = self.masks[index].replace('_leftImg8bit.png', '_gtFine_labelIds.png')  # Adjust mask file naming as needed
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # Assuming mask is grayscale

        if self.transform is not None:
            image = self.transform(image)
            mask = self.transform(mask)

        return {'image':image, 'mask':mask}

    def __len__(self):
        return len(self.images)

dataset = CustomDataset(root,image_dir, mask_dir, transform)
data_loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [2]:
import torch
import torchvision.models as models
import torchvision.models.segmentation as segmentation

# Load pre-trained ResNet-34
resnet34 = models.resnet34(pretrained=True)

# Create a DeepLabv3+ model with ResNet-34 as the backbone
deeplab_model = segmentation.deeplabv3_resnet50(pretrained=False, num_classes=30)

/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [ ]:
!pip install torchmetrics

In [ ]:
import torch.nn as nn
import torch.optim as optim

from torchmetrics.classification import MulticlassJaccardIndex as IoULoss

#os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:256"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()

num_classes = 30
#batch_size = 5
learning_rate = 0.0001
num_epochs = 100

criterion = IoULoss(num_classes = 30).to(device)#nn.CrossEntropyLoss()
optimizer = optim.Adam(deeplab_model.parameters(), lr=learning_rate)

deeplab_model = deeplab_model.to(device)
 
# Training loop
for epoch in range(num_epochs):
    deeplab_model.train()
    for batch in data_loader:
        images = batch['image']
        masks = batch['mask']
        #if torch.cuda.is_available():
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        #print(images.shape)
        outputs = deeplab_model(images)
        
        masks = masks.squeeze(1)
        masks = masks.long()
        #loss = criterion(logits_flat, masks_flat)
        #print(outputs['out'].shape, masks.shape
        #masks = masks.to(device).requires_grad_()
        #print(outputs['out'].requires_grad,masks.requires_grad)
        loss = criterion(outputs['out'], masks)
        loss.requires_grad = True
        loss.backward()
        optimizer.step()
    if epoch%10==0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item()}')
        torch.save(deeplab_model.state_dict(), 'deeplab_model.pth')

Epoch [1/100], Loss: 0.00019727919425349683


In [7]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image
import torch
import torchvision.models.segmentation as segmentation

model = segmentation.deeplabv3_resnet50(pretrained=False, num_classes=30)

model.load_state_dict(torch.load("/kaggle/input/model-pth/deeplab_model.pth"))

model.eval()

/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


DeepLabV3(
  (backbone): IntermediateLayerGetter(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Se

In [8]:
transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
])

def preprocess_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image = transform(image)
    return image

In [9]:
import os

test_images_folder = '/kaggle/input/mapillary/Mapillary-Vistas-1000-sidewalks/testing/images'
output_folder = '/kaggle/working/masks'

os.makedirs(output_folder, exist_ok=True)

for image_name in os.listdir(test_images_folder):
    image_path = os.path.join(test_images_folder, image_name)
    image = preprocess_image(image_path).unsqueeze(0)  # Add batch dimension

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    image = image.to(device)

    model = model.to(device)
    with torch.no_grad():
        output = model(image)

    predicted_mask = (output['out'].argmax(1) == 1).float()
    
    #predicted_mask = torch.nn.functional.softmax(output['out'], dim=1)[0].cpu().numpy()
    #predicted_mask = (predicted_mask * 255).astype('uint8')

    output_path = os.path.join(output_folder, f"{image_name}_predicted_mask.png")
    Image.fromarray((predicted_mask.squeeze().cpu().numpy() * 255).astype('uint8')).save(output_path)
    #Image.fromarray(predicted_mask.transpose(1, 2, 0)).save(output_path)

In [10]:
import numpy as np
import cv2
import os
import argparse

def create_binary_mask(image, rgb_value):
    """
    Create a binary mask from an RGB image based on a specified RGB value.
    
    :param image: Input RGB image
    :param rgb_value: Tuple representing RGB value
    :return: Binary mask
    """
    mask = np.all(image == np.array(rgb_value).reshape(1, 1, 3), axis=2).astype(np.uint8) * 255
    return mask

if __name__ == "__main__":
    input_folder = '/kaggle/input/mapillary/Mapillary-Vistas-1000-sidewalks/testing/labels'
    output_folder = '/kaggle/working/binary_masks'
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    for filename in os.listdir(input_folder):
        if filename.endswith(".jpg") or filename.endswith(".png"):
            image_path = os.path.join(input_folder, filename)
            image = cv2.imread(image_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            if image is not None:
                binary_mask = create_binary_mask(image, tuple([244, 35, 232]))
                
                output_path = os.path.join(output_folder, filename)
                cv2.imwrite(output_path, binary_mask)

    print("Binary masks created successfully!")


Binary masks created successfully!


In [12]:
import os
from PIL import Image
import numpy as np

def compute_iou(predicted_mask_path, ground_truth_mask_path):
    predicted_mask = np.array(Image.open(predicted_mask_path))
    ground_truth_mask = np.array(transform(Image.open(ground_truth_mask_path)))

    intersection = np.logical_and(predicted_mask, ground_truth_mask)
    union = np.logical_or(predicted_mask, ground_truth_mask)

    iou = np.sum(intersection) / np.sum(union)

    return iou

def evaluate_model(predictions_folder, ground_truth_folder):
    predicted_files = os.listdir(predictions_folder)

    total_iou = 0
    total_images = len(predicted_files)

    for file in predicted_files:
        predicted_mask_path = os.path.join(predictions_folder, file)
        ground_truth_mask_path = os.path.join(ground_truth_folder, file.replace(".jpg_predicted_mask.png",".png"))

        iou = compute_iou(predicted_mask_path, ground_truth_mask_path)
        total_iou += iou

    average_iou = total_iou / total_images
    return average_iou

predictions_folder = '/kaggle/working/masks'
ground_truth_folder = '/kaggle/input/mapillary/Mapillary-Vistas-1000-sidewalks/testing/labels'

iou = evaluate_model(predictions_folder, ground_truth_folder)
print(f"Average IoU: {iou}")


Average IoU: 0.16274591125719734
